In [5]:
### 1.Работа с API. Извлечение данных изи OpenAPI T-Invest.
import os
import requests
import re
import logging
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

load_dotenv()

TOKEN = os.getenv('T_TOKEN')
POSTGRES_USER = os.getenv('POSTGRES_USER')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')
POSTGRES_DB = os.getenv('POSTGRES_DB')
URL = 'https://invest-public-api.tbank.ru/rest/tinkoff.public.invest.api.contract.v1.InstrumentsService/GetAssets'

def camel_to_snake(name):
    """Преобразует camelCase в snake_case"""
    # Вставляем _ перед заглавными буквами (кроме первой)
    name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub('([a-z0-9])([A-Z])', r'\1_\2', name)

    return name.lower()

headers = {
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json"
    }

data = {
  "instrumentType": "INSTRUMENT_TYPE_SHARE",
  "instrumentStatus": "INSTRUMENT_STATUS_ALL"
}

response = requests.post(url = URL, headers = headers, json = data, verify = False)

if response.status_code == 200:
    data = response.json()
    df = pd.json_normalize(response.json()['assets'], sep='_', record_path='instruments', meta=['uid', 'name'], meta_prefix='asset_')
else:
    print(f"Ошибка: {response.status_code}")
    print(response.text)
    # return None
    print(None)

# Применяем ко всем столбцам
df.columns = [camel_to_snake(col) for col in df.columns]

/home/kam/datalearnhome/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'invest-public-api.tbank.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [3]:
df[df['figi'] == 'BBG004730N88']

,uid,figi,instrument_type,ticker,class_code,links,instrument_kind,position_uid,asset_uid,asset_name
587,e6123145-9665-43e0-8413-cd61b8aa9b13,BBG004730N88,share,SBER,TQBR,[],INSTRUMENT_TYPE_SHARE,41eb2102-5333-4713-bf15-72b204c4bf7b,40d89385-a03a-4659-bf4e-d3ecba011782,Акции обыкновенные ПАО Сбербанк


In [7]:
data

{'assets': [{'uid': 'b6a73950-20a8-46c7-8b49-9dfbc14fe0ba',
   'type': 'ASSET_TYPE_SECURITY',
   'name': 'Американская депозитарная расписка на обыкновенные акции Rentokil Initial plc',
   'instruments': [{'uid': '44a1e3e8-b813-46dd-b1f8-d0e9d91c06d2',
     'figi': 'BBG000BVPG41',
     'instrumentType': 'share',
     'ticker': 'RTO',
     'classCode': 'SPBXM',
     'links': [],
     'instrumentKind': 'INSTRUMENT_TYPE_SHARE',
     'positionUid': '72161247-8e86-4df5-a911-d06b92620d82'}]},
  {'uid': '1d68d733-8bf8-4565-a7c1-eb9053916b33',
   'type': 'ASSET_TYPE_SECURITY',
   'name': 'Coupa Software Incorporated ORD SHS',
   'instruments': [{'uid': '9b3c9abf-f865-4e62-a49d-8f41626a2160',
     'figi': 'BBG001J4BCN4',
     'instrumentType': 'share',
     'ticker': 'COUP',
     'classCode': 'SPBXM',
     'links': [],
     'instrumentKind': 'INSTRUMENT_TYPE_SHARE',
     'positionUid': '2e23dd2f-b2c7-47e8-b99b-31d16e69ef76'}]},
  {'uid': '899107ef-ad13-4f99-9497-a23afcc67188',
   'type': 'ASSET